# Bench 实验
一格一步，不要 Run All。ESP 每次 reset 时保持线圈电源关闭。结束后运行 `uv run python controller/control/safe_off.py`。

## 0 · 建立 bench
相机顺序 (A, B)、模式、ESP 串口。改任何一项都重跑本格，之前的验证全部作废。

In [ ]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'LLM/bench.py').exists())
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from IPython.display import display, Image
from serial.tools import list_ports
from LLM.bench import Bench, PATTERNS

print('串口:', [p.device for p in list_ports.comports()])
bench = Bench(cameras=(1, 0), serial_port='/dev/cu.usbserial-0001',
              width=1280, height=800, fps=120, upside_down=True)

## 1 · 核对 A/B、对焦

In [ ]:
bench.preview_cameras()

## 2 · 标定（或复用）
重新对焦、单台移动、换模式后设 `NEW_CALIBRATION=True`。板子静止摆多个姿态，按 q 结束。

In [ ]:
NEW_CALIBRATION = False
if NEW_CALIBRATION:
    bench.calibrate()   # = collect_charuco_frames() + calibrate_intrinsics() + calibrate_extrinsics()
else:
    bench.use_calibration(PROJECT/'LLM/runs/setup_002/calibration/stereo_rig.json')

## 3 · 在最终机位验证
至少 5 个新的静止板子姿态，按 q 结束。

In [ ]:
display(bench.verify_calibration())

## 4 · 参考照片
移走板子；LED 与垂线两机都要看得到。点 Save，再 Close。

In [ ]:
reference = bench.photograph_reference()
for tag in 'AB':
    display(Image(filename=str(reference/f'{tag}.png'), width=800))

## 5 · 可选：装置定位
需要 `analyse/fixture_layout.json` 中 `survey_confirmed=true`。没有它，`run()` 做 sync check。

In [ ]:
PLUMB = None
# PLUMB = {'bottom': {'a': [xA, yA], 'b': [xB, yB]},
#          'top':    {'a': [xA, yA], 'b': [xB, yB]}}
if PLUMB:
    display(bench.locate_fixture(PLUMB))

## 6 · 10 秒试录

In [ ]:
display(bench.record())

## 7 · 选择 pattern

In [ ]:
print([p.stem for p in sorted(PATTERNS.glob('*.json'))])
PATTERN = 'rise_2to210Hz_30s_ease_k2'
bench.show_pattern(PATTERN)

### 可选 · 串口握手 / 无录像驱动
`drive_without_cameras` **会驱动线圈**。

In [ ]:
# bench.check_serial()
# bench.drive_without_cameras(PATTERN)

## 8 · 正式实验：Record → START → 自动 stop
**会驱动线圈。** READY 后打开电源。LED ROI 可选（`led_a=[x,y,w,h]`, `led_b=...`）。

In [ ]:
NOTE = ''               # robot / bearing / 线圈、实验目的
WARNINGS_REVIEWED = ''  # 试录有 warnings 时必填核查说明
display(bench.run(PATTERN, note=NOTE, warnings_reviewed=WARNINGS_REVIEWED))

## 9 · 分析：LED 对时 → frequency + 四路 CS 曲线

In [ ]:
result = bench.analyse()
display(result)
for name in ('input_curves.png', 'overview.png'):
    if (bench.last_run/'export'/name).exists():
        display(Image(filename=str(bench.last_run/'export'/name)))